In [1]:
import tensorflow as tf
from tensorflow.keras.layers import (Input, Conv2D, Conv2DTranspose, Dense, Reshape, 
                                     Concatenate, Lambda, BatchNormalization, Add, Multiply, Flatten)
from tensorflow.keras.models import Model
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm  # 주피터용 tqdm
import os

# =========================================================
# 1. 설정 및 상수 (전역 변수)
# =========================================================
# GPU 병렬 처리 전략
strategy = tf.distribute.MirroredStrategy()
print(f"✅ 장치 확인: {strategy.num_replicas_in_sync}개의 GPU를 사용합니다.")

# 배치 사이즈 설정
BATCH_SIZE_PER_REPLICA = 32
GLOBAL_BATCH_SIZE = BATCH_SIZE_PER_REPLICA * strategy.num_replicas_in_sync
IMG_H, IMG_W = 128, 128
LATENT_DIM = 128
EPOCHS_TRANSFER = 100
CHANNELS_TERRAIN = 3
CHANNELS_FIRE = 1
# 파일 경로 (사용자 환경에 맞게 수정)
PATH_GEN_MODEL = './lstm_v_G.keras'
PATH_DISC_MODEL = './lstm_v_D.keras'
TRAIN_TFRECORD = './train_big_128_all_origin_fine.tfrecord'
TEST_TFRECORD = './test_big_128_all_origin_fine.tfrecord'

In [2]:
def _parse_function2(example_proto):
    features = {
        'spread_0h': tf.io.FixedLenFeature([], tf.string),
        'spread_1h': tf.io.FixedLenFeature([], tf.string),
        'spread_2h': tf.io.FixedLenFeature([], tf.string),
        'spread_3h': tf.io.FixedLenFeature([], tf.string),
        'spread_4h': tf.io.FixedLenFeature([], tf.string),
        'spread_5h': tf.io.FixedLenFeature([], tf.string),
        'spread_6h': tf.io.FixedLenFeature([], tf.string),
        'spread_7h': tf.io.FixedLenFeature([], tf.string),
        'spread_8h': tf.io.FixedLenFeature([], tf.string),
        'spread_9h': tf.io.FixedLenFeature([], tf.string),
        'spread_10h': tf.io.FixedLenFeature([], tf.string),
        'spread_11h': tf.io.FixedLenFeature([], tf.string),
        'spread_12h': tf.io.FixedLenFeature([], tf.string),
        'terrain_image': tf.io.FixedLenFeature([], tf.string),
        'dem_image': tf.io.FixedLenFeature([], tf.string),
        'fuel_image': tf.io.FixedLenFeature([], tf.string),
        'tnh': tf.io.FixedLenFeature([], tf.string),
        'wind': tf.io.FixedLenFeature([], tf.string),
        'ignite_index': tf.io.FixedLenFeature([], tf.int64),
        'weather_index': tf.io.FixedLenFeature([], tf.int64)
    }
    parsed = tf.io.parse_single_example(example_proto, features)
    
    def decode_spread(key):
        img = tf.io.decode_raw(parsed[key], tf.float32)
        img = tf.reshape(img, [IMG_H, IMG_W, CHANNELS_FIRE])
        return img
    
    # 우리가 사용할 시간 포인트: 0h, 4h, 8h, 12h
    spread_0h  = decode_spread('spread_0h')
    spread_4h  = decode_spread('spread_4h')
    #spread_4h[spread_4h != 0] = 1
    spread_8h  = decode_spread('spread_8h')
    #spread_8h[spread_8h != 0] = 1
    spread_12h = decode_spread('spread_12h')
    #spread_12h[spread_12h != 0] = 1

    # fire_seq_sub: (4, IMG_H, IMG_W, 1) → 0h는 input, 4h,8h,12h는 타깃 예측용
    fire_seq_sub = tf.stack([spread_0h, spread_4h, spread_8h, spread_12h], axis=0)
    
    # 타깃: target_final은 spread_12h, target_seq는 [spread_4h, spread_8h, spread_12h] → (3, IMG_H, IMG_W, 1)
    target_final = spread_12h
    target_seq = tf.stack([spread_4h, spread_8h, spread_12h], axis=0)
    
    # terrain 이미지: (IMG_H, IMG_W, 3)
    terrain = tf.io.decode_raw(parsed['terrain_image'], tf.float32)
    terrain = tf.reshape(terrain, [IMG_H, IMG_W, CHANNELS_TERRAIN])
    
    # dem, fuel (생략)
    dem = tf.io.decode_raw(parsed['dem_image'], tf.float32)
    dem = tf.reshape(dem, [IMG_H, IMG_W, 1])
    fuel = tf.io.decode_raw(parsed['fuel_image'], tf.float32)
    fuel = tf.reshape(fuel, [IMG_H, IMG_W, 1])
    
    # 기상 데이터: tnh와 wind → (12,4)
    tnh = tf.io.decode_raw(parsed['tnh'], tf.float32)
    tnh = tf.reshape(tnh, [12, 2])
    wind = tf.io.decode_raw(parsed['wind'], tf.float32)
    wind = tf.reshape(wind, [12, 2])
    weather_full = tf.concat([tnh, wind], axis=-1)  # (12,4)
    # 대신 평균 내지 않고, reshape: 12→ 3×4, so weather_seq becomes (3,4,4)
    tnh_seq = tf.reshape(tnh, [3, 4, 2])
    wind_seq = tf.reshape(wind, [3, 4, 2])
    
    ignite_index = parsed['ignite_index']
    weather_index = parsed['weather_index']
    
    # 반환: inputs와 targets
    # inputs: terrain, fire_seq_sub, weather_seq, ignite_index, weather_index
    # targets: target_final, target_seq
    return (terrain, fire_seq_sub, tnh_seq, wind_seq)


def get_dataset(path, batch_size, shuffle=True):
    ds = tf.data.TFRecordDataset(path)
    if shuffle:
        ds = ds.shuffle(buffer_size=4096)
    ds = ds.map(_parse_function2, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=True) # 검증 시 drop_remainder=False 권장
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

# === 데이터셋 로드 (전역 변수로 접근 가능) ===
# 1. 전이학습용 (Shuffle O, Global Batch)
train_ds = get_dataset(TRAIN_TFRECORD, GLOBAL_BATCH_SIZE, shuffle=True)

# 2. Test 평가용 (Shuffle X, Batch 1)
test_ds = get_dataset(TEST_TFRECORD, GLOBAL_BATCH_SIZE, shuffle=False)


print("✅ 데이터셋 준비 완료: train_ds, test_ds, train_eval_ds")

In [3]:
# --- 유틸 함수 ---
def silu(x): return tf.nn.silu(x)

def get_mask(y_true, y_pred, threshold=0.1):
    return tf.cast((y_true > threshold) | (y_pred > threshold), tf.float32)

def masked_mse(y_true, y_pred, mask):
    diff2 = tf.square(y_true - y_pred) * mask
    denom = tf.reduce_sum(mask)
    return tf.cond(denom == 0, lambda: 0.0, lambda: tf.reduce_sum(diff2) / denom)

def calculate_iou(y_true, y_pred, threshold=0.1):
    y_t = tf.cast(y_true > threshold, tf.float32)
    y_p = tf.cast(y_pred > threshold, tf.float32)
    inter = tf.reduce_sum(y_t * y_p)
    union = tf.reduce_sum(y_t) + tf.reduce_sum(y_p) - inter
    return (inter + 1e-7) / (union + 1e-7)

# --- 화선 오차 함수 (수정됨: 3D 입력 처리 + 대칭 정규화) ---
def get_edges(mask):
    # 3D 입력(H,W,C)이 들어오면 4D(1,H,W,C)로 확장
    is_3d = False
    if len(mask.shape) == 3:
        mask = tf.expand_dims(mask, 0)
        is_3d = True
    elif len(mask.shape) == 2:
        mask = tf.expand_dims(mask, 0)
        mask = tf.expand_dims(mask, -1)
        is_3d = True
        
    edge = tf.image.sobel_edges(mask)
    edge_mag = tf.sqrt(tf.reduce_sum(tf.square(edge), axis=-1))
    result = tf.cast(edge_mag > 0.1, tf.float32)
    
    if is_3d: result = tf.squeeze(result, 0)
    return result

def perimeter_rel_err(y_true, y_pred, thr=0.1):
    y_t_bin = tf.cast(y_true > thr, tf.float32)
    y_p_bin = tf.cast(y_pred > thr, tf.float32)
    
    len_t = tf.reduce_sum(get_edges(y_t_bin))
    len_p = tf.reduce_sum(get_edges(y_p_bin))
    
    # Symmetric Relative Error (분모 0 방지 및 안정화)
    return tf.abs(len_t - len_p) / (len_t + len_p + 1e-7)

# --- GAN 클래스 ---
class ProgressiveStepGAN(tf.keras.Model):
    def __init__(self, generator, discriminator, lambda_L1=20.0, lambda_dice=0.5, lambda_gp=10.0, n_critic=4, **kwargs):
        super().__init__(**kwargs)
        self.generator = generator
        self.discriminator = discriminator
        self.lambda_L1, self.lambda_dice, self.lambda_gp, self.n_critic = lambda_L1, lambda_dice, lambda_gp, n_critic

    def compile(self, gen_optimizer, disc_optimizer):
        super().compile()
        self.gen_optimizer = gen_optimizer
        self.disc_optimizer = disc_optimizer

    def gradient_penalty(self, real, fake, inputs):
        (terrain, tnh, wind, step) = inputs
        batch_size = tf.shape(real)[0]
        alpha = tf.random.uniform([batch_size, 1, 1, 1], 0., 1.)
        inter = real * alpha + fake * (1 - alpha)
        with tf.GradientTape() as tape:
            tape.watch(inter)
            pred = self.discriminator([inter, terrain, tnh, wind, step], training=True)
        grads = tape.gradient(pred, inter)
        norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1, 2, 3]) + 1e-12)
        return tf.reduce_mean((norm - 1.0) ** 2)

    def train_step(self, data):
        terrain, fire_seq, tnh_seq, wind_seq = data
        batch_size = tf.shape(terrain)[0]
        idx = tf.random.uniform((batch_size,), 0, 3, dtype=tf.int32)
        step_idx = tf.one_hot(idx, depth=3)
        
        current = tf.gather(fire_seq, idx, axis=1, batch_dims=1)
        target = tf.gather(fire_seq, idx + 1, axis=1, batch_dims=1)
        tnh = tf.gather(tnh_seq, idx, axis=1, batch_dims=1)
        wind = tf.gather(wind_seq, idx, axis=1, batch_dims=1)

        # Disc update
        for _ in range(self.n_critic):
            with tf.GradientTape() as t:
                noise = tf.random.normal((batch_size, LATENT_DIM))
                fake = self.generator([current, terrain, tnh, wind, step_idx, noise], training=False)
                d_real = self.discriminator([target, terrain, tnh, wind, step_idx], training=True)
                d_fake = self.discriminator([fake, terrain, tnh, wind, step_idx], training=True)
                gp = self.gradient_penalty(target, fake, (terrain, tnh, wind, step_idx))
                d_loss = tf.reduce_mean(d_fake) - tf.reduce_mean(d_real) + self.lambda_gp * gp
            self.disc_optimizer.apply_gradients(zip(t.gradient(d_loss, self.discriminator.trainable_variables), self.discriminator.trainable_variables))

        # Gen update
        with tf.GradientTape() as t:
            noise = tf.random.normal((batch_size, LATENT_DIM))
            fake = self.generator([current, terrain, tnh, wind, step_idx, noise], training=True)
            d_fake = self.discriminator([fake, terrain, tnh, wind, step_idx], training=True)
            adv_loss = -tf.reduce_mean(d_fake)
            l1_loss = tf.reduce_mean(tf.abs(target - fake))
            inter = tf.reduce_sum(target * fake, axis=[1,2,3])
            union = tf.reduce_sum(target, axis=[1,2,3]) + tf.reduce_sum(fake, axis=[1,2,3])
            dice_loss = 1.0 - tf.reduce_mean((2.0 * inter + 1e-6) / (union + 1e-6))
            g_loss = adv_loss + self.lambda_L1 * l1_loss + self.lambda_dice * dice_loss
        self.gen_optimizer.apply_gradients(zip(t.gradient(g_loss, self.generator.trainable_variables), self.generator.trainable_variables))
        return {"g_loss": g_loss, "d_loss": d_loss}

custom_objects = {"silu": silu, "ProgressiveStepGAN": ProgressiveStepGAN}
print("✅ 모델 및 함수 정의 완료")

In [39]:
def run_evaluation(model_gen, target_dataset, desc="Eval"):
    # iou 대신 ssim으로 지표 구조 변경
    results = {t: {'mse': [], 'ssim': [], 'perim': []} for t in [4, 8, 12]}
    
    for batch in tqdm(target_dataset, desc=desc):
        terrain, fire_seq, tnh_seq, wind_seq = batch
        B = tf.shape(terrain)[0]
        current = fire_seq[:, 0]
        
        for i in range(3): # 4h, 8h, 12h
            t_hr = (i + 1) * 4
            step = tf.one_hot(tf.fill([B], i), depth=3)
            
            # 단일 예측
            noise = tf.random.normal((B, LATENT_DIM))
            pred = model_gen([current, terrain, tnh_seq[:, i], wind_seq[:, i], step, noise], training=False)
            target = fire_seq[:, i + 1]
            
            # 1. MSE 계산
            mask = get_mask(target, pred)
            results[t_hr]['mse'].append(float(masked_mse(target, pred, mask)))
            
            # 2. SSIM 계산 (iou 대신 적용)
            # tf.image.ssim은 (B, H, W, C) 형태를 지원합니다.
            # 데이터가 [0, 1] 범위로 정규화되어 있다고 가정하여 max_val=1.0 설정
            ssim_values = tf.image.ssim(target, pred, max_val=1.0)
            results[t_hr]['ssim'].append(float(tf.reduce_mean(ssim_values)))
            
            # 3. Perimeter Err 계산 (BMAE)
            p_errs = []
            for b in range(B):
                val = perimeter_rel_err(target[b], pred[b])
                p_errs.append(float(val))
            results[t_hr]['perim'].append(np.mean(p_errs))
            
            current = pred 

    # 최종 평균값 반환
    return {t: {k: np.mean(v) for k, v in results[t].items()} for t in [4, 8, 12]}

def print_results(summary, title="Results"):
    print(f"\n=== {title} ===")
    print(f"{'Time':<6} | {'MSE':<10} | {'SSIM':<10} | {'PerimErr':<10}")
    print("-" * 45)
    for t in [4, 8, 12]:
        s = summary[t]
        print(f"{t:>2}hr   | {s['mse']:.5f}    | {s['ssim']:.5f}    | {s['perim']:.5f}")

In [ ]:
with strategy.scope():
    print("🔄 모델 로드 중...")
    # 1. 모델 로드
    gen_base = tf.keras.models.load_model(PATH_GEN_MODEL, custom_objects=custom_objects, compile=False)
    disc_base = tf.keras.models.load_model(PATH_DISC_MODEL, custom_objects=custom_objects, compile=False)
    
    # 2. 비교용 복제본 생성 (CGAN-baseded Weights 저장)
    gen_old = tf.keras.models.clone_model(gen_base)
    gen_old.set_weights(gen_base.get_weights())

# Baseline 평가
print("📊 [Baseline] Test Set 평가...")
res_pre_test = run_evaluation(gen_base, test_ds, desc="Test Eval")
print_results(res_pre_test, "CGAN-based Test Results")

print("📊 [Baseline] Train Set 평가 (Subset)...")
res_pre_train = run_evaluation(gen_base, train_ds, desc="Train Eval")
print_results(res_pre_train, "CGAN-based Train Results")

In [ ]:
with strategy.scope():
    # 1. Encoder 동결
    print("🔒 Encoder 동결 및 Optimizer 설정...")
    for layer in gen_base.layers:
        if 'conv2d_6' in layer.name or 'batch' in layer.name:
            if 'transpose' not in layer.name: # Decoder는 학습
                layer.trainable = False
        else: 
            layer.trainable = True
            
    # 2. GAN 컴파일
    gan = ProgressiveStepGAN(gen_base, disc_base)
    gan.compile(
        gen_optimizer=tf.keras.optimizers.Adam(1e-5, beta_1 = 0.5), # Low Learning Rate
        disc_optimizer=tf.keras.optimizers.Adam(2e-5, beta_1 = 0.5)
    )



In [19]:
print(f"🔥 전이학습 시작 (Epochs: {200})...")
history = gan.fit(train_ds, epochs=100)
print("✅ 학습 완료!")

In [ ]:
# Baseline 평가
print("📊 [Baseline] Test Set 평가...")
res_pre_test = run_evaluation(gen_old, test_ds, desc="Test Eval")
print_results(res_pre_test, "CGAN-based Test Results")

print("📊 [Baseline] Train Set 평가 (Subset)...")
res_pre_train = run_evaluation(gen_old, train_ds, desc="Train Eval")
print_results(res_pre_train, "CGAN-based Train Results")

print("📊 [TL-CGAN-based model] Test Set 평가...")
res_post_test = run_evaluation(gan.generator, test_ds, desc="Post-Test Eval")
print_results(res_post_test, "TL-CGAN-based model Test Results")

print("📊 [TL-CGAN-based model] Train Set 평가...")
res_post_train = run_evaluation(gan.generator, train_ds, desc="TL-CGAN-based model Eval")
print_results(res_post_train, "TL-CGAN-based model Train Results")

# 간단 비교 출력
iou_imp = (res_post_test[12]['SSIM'] - res_pre_test[12]['SSIM']) / res_pre_test[12]['SSIM'] * 100
print(f"\n🚀 Test IoU Improvement (12h): {iou_imp:.2f}%")

In [21]:
def visualize_batch_samples(gen_pre, gen_post, dataset, num_samples=4, skip_batches=0):
    """
    배치 사이즈가 커도(16 등) 내부에서 인덱싱하여 여러 결과를 동시에 시각화합니다.
    
    Args:
        gen_pre: 학습 전 모델 (Generator)
        gen_post: 학습 후 모델 (Generator)
        dataset: 데이터셋 (Batch Size > 1 권장)
        num_samples: 시각화할 샘플 개수 (배치 사이즈보다 작아야 함)
        skip_batches: 몇 번째 배치부터 볼 것인지 건너뛰기
    """
    # 1. 원하는 배치만큼 건너뛰고 데이터 가져오기
    for batch in dataset.skip(skip_batches).take(1):
        terrain, fire_seq, tnh_seq, wind_seq = batch
        break
    
    # 2. 배치 내 실제 데이터 개수 확인 (마지막 배치는 작을 수 있음)
    actual_batch_size = tf.shape(terrain)[0]
    n_display = min(num_samples, actual_batch_size)
    
    print(f"🔍 배치 내 {actual_batch_size}개 데이터 중 {n_display}개를 시각화합니다.")

    # 3. 모델 예측 (배치 단위로 한 번에 수행)
    # 초기 입력 (0h)
    curr_pre = fire_seq[:, 0]
    curr_post = fire_seq[:, 0]
    
    # 3단계(12h)까지 재귀적 예측
    for i in range(3):
        # Step One-hot 인코딩 (배치 크기에 맞게 복제)
        # step shape: (B, 3)
        step_idx = tf.one_hot(tf.fill([actual_batch_size], i), depth=3)
        
        # Noise (B, 128)
        noise = tf.random.normal((actual_batch_size, LATENT_DIM))
        
        # 기상 데이터 (B, 4, 2)
        tnh_step = tnh_seq[:, i]
        wind_step = wind_seq[:, i]
        
        # 예측 (Training=False)
        curr_pre = gen_pre([curr_pre, terrain, tnh_step, wind_step, step_idx, noise], training=False)
        curr_post = gen_post([curr_post, terrain, tnh_step, wind_step, step_idx, noise], training=False)

    # 4. 시각화 (Grid 형태)
    # 행: 샘플 개수, 열: 4개 (지형, 정답, 전, 후)
    fig, axes = plt.subplots(n_display, 4, figsize=(16, 4 * n_display))
    
    # 샘플이 1개일 경우 axes가 1차원 배열이 되므로 2차원으로 변환
    if n_display == 1:
        axes = axes[np.newaxis, :]

    for idx in range(n_display):
        # A. 지형 (RGB)
        axes[idx, 0].imshow(terrain[idx, :, :, :3])
        axes[idx, 0].set_title(f"Sample {idx+1}: Input Terrain")
        axes[idx, 0].axis('off')
        
        # B. 정답 (12h)
        target = fire_seq[idx, 3, :, :, 0]
        axes[idx, 1].imshow(target, cmap='inferno')
        axes[idx, 1].set_title("Ground Truth (12h)")
        axes[idx, 1].axis('off')
        
        # C. 학습 전
        pred_pre = curr_pre[idx, :, :, 0]
        iou_pre = calculate_iou(target, pred_pre)
        axes[idx, 2].imshow(pred_pre, cmap='inferno')
        axes[idx, 2].set_title(f"Before Transfer\nIoU: {iou_pre:.3f}")
        axes[idx, 2].axis('off')
        
        # D. 학습 후
        pred_post = curr_post[idx, :, :, 0]
        iou_post = calculate_iou(target, pred_post)
        axes[idx, 3].imshow(pred_post, cmap='inferno')
        axes[idx, 3].set_title(f"After Transfer\nIoU: {iou_post:.3f}")
        axes[idx, 3].axis('off')
        
        # IoU 비교 텍스트 (상승폭 표시)
        diff = iou_post - iou_pre
        color = 'green' if diff > 0 else 'red'
        axes[idx, 3].text(5, 120, f"{diff:+.3f}", color=color, fontweight='bold', 
                          bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

    plt.tight_layout()
    plt.show()

# === 실행 예시 ===
# batch_size=16인 train_ds나 test_ds를 그대로 넣으시면 됩니다.
# num_samples=5: 한 번에 5개씩 비교
# skip_batches=0: 첫 번째 배치, 1: 두 번째 배치 ...
#visualize_batch_samples(gen_pre_trained, gan.generator, test_ds, num_samples=5, skip_batches=0)

# 사용 예시: 0번째, 5번째, 10번째 데이터 확인

for i in range(3):
    visualize_batch_samples(gen_old, gan.generator, test_ds, num_samples=10, skip_batches=i)
# visualize_sample(gen_old, gan.generator, test_ds, skip_count=5)

In [69]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

def visualize_best_improved_samples(gen_pre, gen_post, dataset, num_displays=3, num_batches_to_search=10):
    """
    IoU 상승 폭이 가장 큰 샘플을 자동으로 찾아 3x3 레이아웃으로 시각화합니다.
    
    Args:
        gen_pre: 전이학습 전 모델 (Generator)
        gen_post: 전이학습 후 모델 (Generator)
        dataset: 데이터셋
        num_displays: 시각화할 샘플의 개수 (상위 N개)
        num_batches_to_search: 최적의 샘플을 찾기 위해 검색할 배치 수
    """
    all_samples_info = []

    # 1. 최적의 샘플 검색 (여러 배치를 돌며 IoU 개선 폭 계산)
    for batch in dataset.take(num_batches_to_search):
        terrain, fire_seq, tnh_seq, wind_seq = batch
        batch_size = tf.shape(terrain)[0]
        
        # 예측값 저장용 (B, 3, H, W, 1) -> 4h, 8h, 12h
        preds_pre = []
        preds_post = []
        
        curr_pre = fire_seq[:, 0]
        curr_post = fire_seq[:, 0]
        
        # 3단계 재귀적 예측 (4h, 8h, 12h)
        for i in range(3):
            step_idx = tf.one_hot(tf.fill([batch_size], i), depth=3)
            noise = tf.random.normal((batch_size, LATENT_DIM))
            tnh_step = tnh_seq[:, i]
            wind_step = wind_seq[:, i]
            
            curr_pre = gen_pre([curr_pre, terrain, tnh_step, wind_step, step_idx, noise], training=False)
            curr_post = gen_post([curr_post, terrain, tnh_step, wind_step, step_idx, noise], training=False)
            
            preds_pre.append(curr_pre.numpy())
            preds_post.append(curr_post.numpy())
        
        # 12h(Index 2) 기준 IoU 계산 및 개선폭 저장
        for b in range(batch_size):
            target_12h = fire_seq[b, 3, :, :, 0].numpy()
            iou_pre = calculate_iou(target_12h, preds_pre[2][b, :, :, 0])
            iou_post = calculate_iou(target_12h, preds_post[2][b, :, :, 0])
            improvement = iou_post - iou_pre
            
            all_samples_info.append({
                'improvement': improvement,
                'iou_pre': iou_pre,
                'iou_post': iou_post,
                'terrain': terrain[b].numpy(),
                'targets': fire_seq[b, 1:4].numpy(), # 4h, 8h, 12h GT
                'preds_pre': [preds_pre[0][b], preds_pre[1][b], preds_pre[2][b]],
                'preds_post': [preds_post[0][b], preds_post[1][b], preds_post[2][b]]
            })

    # 2. 개선 폭(improvement) 기준 내림차순 정렬하여 상위 샘플 선택
    all_samples_info.sort(key=lambda x: x['improvement'], reverse=True)
    selected_samples = all_samples_info[:num_displays]

    # 3. 시각화 (각 샘플당 3x3 그리드 생성)
    for s_idx, sample in enumerate(selected_samples):
        # 좌측 레이블 공간을 위해 figsize 너비를 조금 넓힙니다.
        fig, axes = plt.subplots(3, 3, figsize=(12, 11), facecolor='white')
        
        col_labels = ['T = 4hr', 'T = 8hr', 'T = 12hr']
        # 두 번째 이미지와 유사한 레이블 명칭으로 설정
        row_labels = ['Ground Truth', 'Predicted Spread\n(TL-CGAN-based)', 'Predicted Spread\n(CGAN-based)']
        
        for t_step in range(3):
            # 데이터 추출
            gt_img = sample['targets'][t_step, :, :, 0]
            post_img = sample['preds_post'][t_step][:, :, 0]
            pre_img = sample['preds_pre'][t_step][:, :, 0]
            
            # 1. 시각화 (타이틀 제거)
            axes[0, t_step].imshow(gt_img, cmap='hot')
            axes[1, t_step].imshow(post_img, cmap='hot')
            axes[2, t_step].imshow(pre_img, cmap='hot')
            
            # 2. 모든 축 눈금 및 타이틀 제거
            for r in range(3):
                axes[r, t_step].set_xticks([])
                axes[r, t_step].set_yticks([])
                # 개별 타이틀을 지웁니다.
                axes[r, t_step].set_title("") 

            # 3. 좌측 행 레이블 추가 (첫 번째 열에만 표시)
            if t_step == 0:
                for r in range(3):
                    axes[r, 0].text(-0.1, 0.5, row_labels[r], 
                                    transform=axes[r, 0].transAxes, 
                                    fontsize=18,  
                                    va='center', ha='right')

            # 4. 하단 열 레이블 추가 (마지막 행에만 표시)
            axes[2, t_step].text(0.5, -0.15, col_labels[t_step], 
                                transform=axes[2, t_step].transAxes, 
                                fontsize=16, 
                                va='top', ha='center')
        
        # 여백 조정 (이미지 스타일처럼 촘촘하게)
        plt.subplots_adjust(wspace=0.05, hspace=0.05, left=0.2, bottom=0.1)
        
        plt.savefig(f'wildfire_comparison_styled_{s_idx+1}.png', bbox_inches='tight', dpi=300)
        plt.show()

# 실행
# LATENT_DIM 등은 기존 코드의 전역 변수를 사용한다고 가정합니다.
visualize_best_improved_samples(gen_old, gan.generator, test_ds, num_displays=100)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import (Input, Conv2D, Conv2DTranspose, Dense, Reshape, 
                                     Concatenate, Lambda, BatchNormalization, Add, Multiply, Flatten)
from tensorflow.keras.models import Model
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

strategy = tf.distribute.MirroredStrategy()
print(f"✅ 장치 확인: {strategy.num_replicas_in_sync}개의 GPU를 사용합니다.")

# =========================================================
# 1. 설정 및 상수
# =========================================================
IMG_H, IMG_W = 128, 128
LATENT_DIM = 128
BATCH_SIZE_PER_REPLICA = 32
# 전체 배치 사이즈 = GPU 개수 * 개당 배치 사이즈
GLOBAL_BATCH_SIZE = BATCH_SIZE_PER_REPLICA * strategy.num_replicas_in_sync
EPOCHS_TRANSFER = 100  # 전이학습 에포크 (데이터 양에 따라 조절)
CHANNELS_TERRAIN = 3
CHANNELS_FIRE = 1

WEATHER_TNH_SHAPE = (4,2)
WEATHER_WIND_SHAPE = (4,2)
TIME_STEPS_MAX = 3
# 파일 경로 (사용자 환경에 맞게 수정 필수)
PATH_GEN_MODEL = './lstm_v_G.keras'      # 저장된 Generator 경로
PATH_DISC_MODEL = './lstm_v_D.keras' # 저장된 Discriminator 경로
TRAIN_TFRECORD = './train_big_128_all_origin_fine.tfrecord'       # 튜닝된 FARSITE 학습 데이터
TEST_TFRECORD = './test_big_128_all_origin_fine.tfrecord'         # 튜닝된 FARSITE 테스트 데이터


# =========================================================
# 2. 데이터셋 로드 (파싱 함수 포함)
# =========================================================
def _parse_function2(example_proto):
    features = {
        'spread_0h': tf.io.FixedLenFeature([], tf.string),
        'spread_1h': tf.io.FixedLenFeature([], tf.string),
        'spread_2h': tf.io.FixedLenFeature([], tf.string),
        'spread_3h': tf.io.FixedLenFeature([], tf.string),
        'spread_4h': tf.io.FixedLenFeature([], tf.string),
        'spread_5h': tf.io.FixedLenFeature([], tf.string),
        'spread_6h': tf.io.FixedLenFeature([], tf.string),
        'spread_7h': tf.io.FixedLenFeature([], tf.string),
        'spread_8h': tf.io.FixedLenFeature([], tf.string),
        'spread_9h': tf.io.FixedLenFeature([], tf.string),
        'spread_10h': tf.io.FixedLenFeature([], tf.string),
        'spread_11h': tf.io.FixedLenFeature([], tf.string),
        'spread_12h': tf.io.FixedLenFeature([], tf.string),
        'terrain_image': tf.io.FixedLenFeature([], tf.string),
        'dem_image': tf.io.FixedLenFeature([], tf.string),
        'fuel_image': tf.io.FixedLenFeature([], tf.string),
        'tnh': tf.io.FixedLenFeature([], tf.string),
        'wind': tf.io.FixedLenFeature([], tf.string),
        'ignite_index': tf.io.FixedLenFeature([], tf.int64),
        'weather_index': tf.io.FixedLenFeature([], tf.int64)
    }
    parsed = tf.io.parse_single_example(example_proto, features)
    
    def decode_spread(key):
        img = tf.io.decode_raw(parsed[key], tf.float32)
        img = tf.reshape(img, [IMG_H, IMG_W, CHANNELS_FIRE])
        return img
    
    # 우리가 사용할 시간 포인트: 0h, 4h, 8h, 12h
    spread_0h  = decode_spread('spread_0h')
    spread_4h  = decode_spread('spread_4h')
    #spread_4h[spread_4h != 0] = 1
    spread_8h  = decode_spread('spread_8h')
    #spread_8h[spread_8h != 0] = 1
    spread_12h = decode_spread('spread_12h')
    #spread_12h[spread_12h != 0] = 1

    # fire_seq_sub: (4, IMG_H, IMG_W, 1) → 0h는 input, 4h,8h,12h는 타깃 예측용
    fire_seq_sub = tf.stack([spread_0h, spread_4h, spread_8h, spread_12h], axis=0)
    
    # 타깃: target_final은 spread_12h, target_seq는 [spread_4h, spread_8h, spread_12h] → (3, IMG_H, IMG_W, 1)
    target_final = spread_12h
    target_seq = tf.stack([spread_4h, spread_8h, spread_12h], axis=0)
    
    # terrain 이미지: (IMG_H, IMG_W, 3)
    terrain = tf.io.decode_raw(parsed['terrain_image'], tf.float32)
    terrain = tf.reshape(terrain, [IMG_H, IMG_W, CHANNELS_TERRAIN])
    
    # dem, fuel (생략)
    dem = tf.io.decode_raw(parsed['dem_image'], tf.float32)
    dem = tf.reshape(dem, [IMG_H, IMG_W, 1])
    fuel = tf.io.decode_raw(parsed['fuel_image'], tf.float32)
    fuel = tf.reshape(fuel, [IMG_H, IMG_W, 1])
    
    # 기상 데이터: tnh와 wind → (12,4)
    tnh = tf.io.decode_raw(parsed['tnh'], tf.float32)
    tnh = tf.reshape(tnh, [12, 2])
    wind = tf.io.decode_raw(parsed['wind'], tf.float32)
    wind = tf.reshape(wind, [12, 2])
    weather_full = tf.concat([tnh, wind], axis=-1)  # (12,4)
    # 대신 평균 내지 않고, reshape: 12→ 3×4, so weather_seq becomes (3,4,4)
    tnh_seq = tf.reshape(tnh, [3, 4, 2])
    wind_seq = tf.reshape(wind, [3, 4, 2])
    
    ignite_index = parsed['ignite_index']
    weather_index = parsed['weather_index']
    
    # 반환: inputs와 targets
    # inputs: terrain, fire_seq_sub, weather_seq, ignite_index, weather_index
    # targets: target_final, target_seq
    return (terrain, fire_seq_sub, tnh_seq, wind_seq)

def get_dataset(path, batch_size=16):
    ds = tf.data.TFRecordDataset(path)
    ds = ds.shuffle(buffer_size=4096)
    ds = ds.map(_parse_function2, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
    return ds

# =========================================================
# 3. 모델 및 유틸리티 클래스
# =========================================================
def silu(x): return tf.nn.silu(x)

def get_mask(y_true, y_pred, threshold=0.1):
    return tf.cast((y_true > threshold) | (y_pred > threshold), tf.float32)

def masked_mse(y_true, y_pred, mask):
    diff2 = tf.square(y_true - y_pred) * mask
    denom = tf.reduce_sum(mask)
    return tf.cond(denom == 0, lambda: 0.0, lambda: tf.reduce_sum(diff2) / denom)

def calculate_iou(y_true, y_pred, threshold=0.1):
    y_t = tf.cast(y_true > threshold, tf.float32)
    y_p = tf.cast(y_pred > threshold, tf.float32)
    inter = tf.reduce_sum(y_t * y_p)
    union = tf.reduce_sum(y_t) + tf.reduce_sum(y_p) - inter
    return (inter + 1e-7) / (union + 1e-7)

# 화선 오차 계산 함수
def get_edges(mask):
    """
    Sobel Edge 검출 함수 (차원 호환성 수정)
    mask: (H, W, C) 또는 (B, H, W, C)
    """
    # 1. 차원 확인 및 확장 (3D -> 4D)
    # tf.image.sobel_edges는 반드시 4차원 입력을 요구합니다.
    is_3d = False
    if len(mask.shape) == 3:
        mask = tf.expand_dims(mask, 0) # (H,W,C) -> (1,H,W,C)
        is_3d = True
    elif len(mask.shape) == 2:
        mask = tf.expand_dims(mask, -1)
        mask = tf.expand_dims(mask, 0) # (H,W) -> (1,H,W,1)
        is_3d = True
        
    # 2. 엣지 검출
    edge = tf.image.sobel_edges(mask)
    edge_mag = tf.sqrt(tf.reduce_sum(tf.square(edge), axis=-1))
    
    # 이진화
    result = tf.cast(edge_mag > 0.1, tf.float32)
    
    # 3. 차원 복원 (입력이 3D였으면 다시 3D로)
    if is_3d:
        result = tf.squeeze(result, 0)
        
    return result

def perimeter_rel_err(y_true, y_pred, thr=0.1):
    """
    화선 길이 오차율 계산 (Symmetric 방식 적용)
    - 결과가 0~1 사이로 정규화되어 초기 화재(4h)에서도 값이 튀지 않음
    """
    y_t_bin = tf.cast(y_true > thr, tf.float32)
    y_p_bin = tf.cast(y_pred > thr, tf.float32)
    
    # 엣지 길이 추출
    len_t = tf.reduce_sum(get_edges(y_t_bin))
    len_p = tf.reduce_sum(get_edges(y_p_bin))
    
    # [핵심 수정] 분모를 (정답 + 예측)으로 변경하여 0 division 방지 및 정규화
    # 둘 다 0이면 오차 0
    numerator = tf.abs(len_t - len_p)
    denominator = len_t + len_p + 1e-7
    
    return numerator / denominator

class ProgressiveStepGAN(tf.keras.Model):
    def __init__(self, generator, discriminator, lambda_L1=20.0, lambda_dice=0.5, lambda_gp=10.0, n_critic=4, **kwargs):
        super(ProgressiveStepGAN, self).__init__(**kwargs)
        self.generator = generator
        self.discriminator = discriminator
        self.lambda_L1, self.lambda_dice, self.lambda_gp, self.n_critic = lambda_L1, lambda_dice, lambda_gp, n_critic

    def compile(self, gen_optimizer, disc_optimizer):
        super(ProgressiveStepGAN, self).compile()
        self.gen_optimizer = gen_optimizer
        self.disc_optimizer = disc_optimizer

    def gradient_penalty(self, real, fake, inputs):
        (terrain, tnh, wind, step) = inputs
        batch_size = tf.shape(real)[0] # 동적 배치 사이즈
        alpha = tf.random.uniform([batch_size, 1, 1, 1], 0., 1.)
        inter = real * alpha + fake * (1 - alpha)
        with tf.GradientTape() as tape:
            tape.watch(inter)
            pred = self.discriminator([inter, terrain, tnh, wind, step], training=True)
        grads = tape.gradient(pred, inter)
        norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1, 2, 3]) + 1e-12)
        return tf.reduce_mean((norm - 1.0) ** 2)

    def train_step(self, data):
        terrain, fire_seq, tnh_seq, wind_seq = data
        batch_size = tf.shape(terrain)[0]
        
        idx = tf.random.uniform((batch_size,), 0, 3, dtype=tf.int32)
        step_idx = tf.one_hot(idx, depth=3)
        
        current = tf.gather(fire_seq, idx, axis=1, batch_dims=1)
        target = tf.gather(fire_seq, idx + 1, axis=1, batch_dims=1)
        tnh = tf.gather(tnh_seq, idx, axis=1, batch_dims=1)
        wind = tf.gather(wind_seq, idx, axis=1, batch_dims=1)

        # 1. Discriminator Update
        for _ in range(self.n_critic):
            with tf.GradientTape() as t:
                noise = tf.random.normal((batch_size, LATENT_DIM))
                fake = self.generator([current, terrain, tnh, wind, step_idx, noise], training=False)
                d_real = self.discriminator([target, terrain, tnh, wind, step_idx], training=True)
                d_fake = self.discriminator([fake, terrain, tnh, wind, step_idx], training=True)
                
                gp = self.gradient_penalty(target, fake, (terrain, tnh, wind, step_idx))
                d_loss = tf.reduce_mean(d_fake) - tf.reduce_mean(d_real) + self.lambda_gp * gp
            
            grads = t.gradient(d_loss, self.discriminator.trainable_variables)
            self.disc_optimizer.apply_gradients(zip(grads, self.discriminator.trainable_variables))

        # 2. Generator Update
        with tf.GradientTape() as t:
            noise = tf.random.normal((batch_size, LATENT_DIM))
            fake = self.generator([current, terrain, tnh, wind, step_idx, noise], training=True)
            d_fake = self.discriminator([fake, terrain, tnh, wind, step_idx], training=True)
            
            adv_loss = -tf.reduce_mean(d_fake)
            l1_loss = tf.reduce_mean(tf.abs(target - fake))
            
            inter = tf.reduce_sum(target * fake, axis=[1,2,3])
            union = tf.reduce_sum(target, axis=[1,2,3]) + tf.reduce_sum(fake, axis=[1,2,3])
            dice_loss = 1.0 - tf.reduce_mean((2.0 * inter + 1e-6) / (union + 1e-6))
            
            g_loss = adv_loss + self.lambda_L1 * l1_loss + self.lambda_dice * dice_loss
            
        grads = t.gradient(g_loss, self.generator.trainable_variables)
        self.gen_optimizer.apply_gradients(zip(grads, self.generator.trainable_variables))
        
        return {"g_loss": g_loss, "d_loss": d_loss, "iou": calculate_iou(target, fake)}

# =========================================================
# 4. 검증 함수 (AttributeError 해결 버전)
# =========================================================
def evaluate_stepwise(generator, dataset, n_ensemble=5):
    results = {t: {'mse': [], 'iou': [], 'perim': []} for t in [4, 8, 12]}
    
    for batch in tqdm(dataset, desc="Evaluating"):
        terrain, fire_seq, tnh_seq, wind_seq = batch
        B = tf.shape(terrain)[0]
        current = fire_seq[:, 0]
        
        for i in range(3):
            t_hr = (i + 1) * 4
            step = tf.one_hot(tf.fill([B], i), depth=3)
            
            # 앙상블 예측
            preds = []
            for _ in range(n_ensemble):
                noise = tf.random.normal((B, LATENT_DIM))
                p = generator([current, terrain, tnh_seq[:, i], wind_seq[:, i], step, noise], training=False)
                preds.append(p)
            mean_pred = tf.reduce_mean(tf.stack(preds), axis=0)
            target = fire_seq[:, i + 1]
            
            # MSE & IoU
            mask = get_mask(target, mean_pred)
            # float()로 변환하여 저장
            results[t_hr]['mse'].append(float(masked_mse(target, mean_pred, mask)))
            results[t_hr]['iou'].append(float(calculate_iou(target, mean_pred)))
            
            # Perimeter Err (배치별 루프)
            p_errs = []
            for b in range(B):
                val = perimeter_rel_err(target[b], mean_pred[b])
                p_errs.append(float(val))
            
            # 배치 평균을 리스트에 추가
            results[t_hr]['perim'].append(np.mean(p_errs))
            
            current = mean_pred 

    # 최종 평균 계산
    return {t: {k: np.mean(v) for k, v in results[t].items()} for t in [4, 8, 12]}

def visualize_comparison(gen_pre, gen_post, dataset, save_path="rebuttal_result.png"):
    for batch in dataset.take(1):
        terrain, fire_seq, tnh_seq, wind_seq = batch
        break
    idx = 0
    curr_pre = fire_seq[idx:idx+1, 0]
    curr_post = fire_seq[idx:idx+1, 0]
    
    for i in range(3):
        step = tf.one_hot([i], depth=3)
        noise = tf.random.normal((1, LATENT_DIM))
        curr_pre = gen_pre([curr_pre, terrain[idx:idx+1], tnh_seq[idx:idx+1, i], wind_seq[idx:idx+1, i], step, noise], training=False)
        curr_post = gen_post([curr_post, terrain[idx:idx+1], tnh_seq[idx:idx+1, i], wind_seq[idx:idx+1, i], step, noise], training=False)
    
    target_12h = fire_seq[idx, 3, :, :, 0]
    plt.figure(figsize=(20, 5))
    plt.subplot(1, 4, 1); plt.title("Input (Terrain)"); plt.imshow(terrain[idx, :, :, :3]); plt.axis('off')
    plt.subplot(1, 4, 2); plt.title("Ground Truth (12h)"); plt.imshow(target_12h, cmap='inferno'); plt.axis('off')
    plt.subplot(1, 4, 3); plt.title("Before Transfer"); plt.imshow(curr_pre[0,:,:,0], cmap='inferno'); plt.axis('off')
    plt.subplot(1, 4, 4); plt.title("After Transfer"); plt.imshow(curr_post[0,:,:,0], cmap='inferno'); plt.axis('off')
    plt.tight_layout(); plt.savefig(save_path, dpi=300); plt.show()

# =========================================================
# 5. 메인 실행 함수 (Multi-GPU 적용)
# =========================================================
def main():
    print("🚀 프로세스 시작...")
    
    # 1. 데이터셋 준비 (GLOBAL_BATCH_SIZE 사용)
    # TFRecordDataset은 Strategy 외부에서 생성해도 무방하나, batch는 global size여야 함
    train_ds = get_dataset(TRAIN_TFRECORD, GLOBAL_BATCH_SIZE)
    test_ds = get_dataset(TEST_TFRECORD, 1) # 검증은 batch=1

    # 2. 모델 로드 및 Strategy Scope 적용
    print("\n🔄 모델 로드 및 Multi-GPU 초기화...")
    custom_objects = {"silu": silu, "ProgressiveStepGAN": ProgressiveStepGAN}
    
    # [중요] 모델 생성/로드 및 컴파일은 반드시 strategy.scope() 안에서 수행!
    with strategy.scope():
        # A. Baseline 모델 로드 (평가용, 가중치 고정)
        # load_model은 scope 안에서 호출하면 자동으로 복제됨
        gen_base = tf.keras.models.load_model(PATH_GEN_MODEL, custom_objects=custom_objects, compile=False)
        disc_base = tf.keras.models.load_model(PATH_DISC_MODEL, custom_objects=custom_objects, compile=False)
        
        # B. 전이학습용 모델 설정
        # 모델을 복제하지 않고 gen_base를 직접 학습시키면 baseline 평가가 오염되므로
        # gen_base는 평가용으로 두고, 학습용 gan_model에 들어갈 모델은 새로 로드하거나 복제
        # (여기서는 메모리 절약을 위해 gen_base를 평가 후 학습 모드로 전환하는 방식 사용)
        
        # --- [Step 1] 전이학습 전 평가 (Scope 안에서도 평가 가능) ---
        print("\n📊 전이학습 전 성능 평가...")
        # 평가는 gen_base(학습 전 상태)로 수행
        # 주의: evaluate_stepwise 함수 내부의 연산도 GPU에서 수행됨
        res_pre = evaluate_stepwise(gen_base, test_ds)
        
        # --- 전이학습 준비 ---
        print("\n🔒 Encoder 동결 및 Optimizer 설정...")
        for layer in gen_base.layers:
            # Encoder 부분 동결
            if 'conv2d_6' in layer.name or 'batch' in layer.name: 
                if 'transpose' not in layer.name: layer.trainable = False
            else: layer.trainable = True
        
        # GAN 통합 및 컴파일 (Scope 내부 필수)
        gan = ProgressiveStepGAN(gen_base, disc_base)
        gan.compile(
            gen_optimizer=tf.keras.optimizers.Adam(1e-5),
            disc_optimizer=tf.keras.optimizers.Adam(2e-5)
        )

    # 3. [Step 2] 전이학습 실행 (Multi-GPU)
    # strategy.scope() 밖에서 .fit()을 호출해도, 모델이 scope 안에서 컴파일되었다면 분산 처리됨
    print(f"\n🔥 전이학습 시작 (Epochs: {EPOCHS_TRANSFER}, Global Batch: {GLOBAL_BATCH_SIZE})...")
    gan.fit(train_ds, epochs=EPOCHS_TRANSFER)

    # 4. [Step 3] 최종 평가
    print("\n📊 전이학습 후 성능 평가...")
    res_post = evaluate_stepwise(gan.generator, test_ds)

    # 5. 결과 출력
    print("\n" + "="*70)
    print(f"{'Time':<6} | {'Metric':<12} | {'Before FT':<12} | {'After FT':<12} | {'Improve'}")
    print("-" * 70)
    for t in [4, 8, 12]:
        for k in ['mse', 'iou', 'perim']:
            v_pre, v_post = res_pre[t][k], res_post[t][k]
            imp = (v_post - v_pre) / v_pre * 100 if k == 'iou' else (v_pre - v_post) / v_pre * 100
            print(f"{t:>2}hr   | {k:<12} | {v_pre:<12.5f} | {v_post:<12.5f} | {imp:>6.2f}%")
    print("="*70)
    
    # 6. 시각화 (시각화용 모델 복제는 메모리 이슈로 생략, 현재 학습된 모델로 결과 출력)
    # 비교를 위해 학습 전 결과(curr_pre)를 얻으려면 학습 전 가중치를 따로 저장했어야 함
    # 여기서는 학습된 모델(After)과 GT만 비교하거나, 위 코드 흐름상 res_pre 값으로 만족
    print("\n🎨 결과 시각화 (After Transfer vs GT)...")
    visualize_comparison(gan.generator, gan.generator, test_ds) # Pre/Post 비교 시각화는 별도 가중치 로드 필요

if __name__ == "__main__":
    main()


In [ ]:
def analyze_ensemble_impact(generator, dataset, ensemble_steps=[1, 3, 5, 10, 15, 20]):
    """
    앙상블 횟수(N)를 늘려가며 성능(IoU, MSE)이 어떻게 변하는지 정량적으로 분석
    """
    # 최대 횟수만큼 미리 예측을 모으기 위해 가장 큰 값 추출
    max_ens = max(ensemble_steps)
    
    # 결과 저장용: {1: [], 5: [], ...}
    results = {n: {'iou': [], 'mse': []} for n in ensemble_steps}
    
    print(f"📊 앙상블 분석 시작 (Max Ensemble: {max_ens})...")
    
    for batch in tqdm(dataset, desc="Analyzing Ensemble"):
        terrain, fire_seq, tnh_seq, wind_seq = batch
        batch_size = tf.shape(terrain)[0]
        
        # --- [Step 1] 최대 횟수(max_ens)만큼 예측 생성 및 저장 ---
        # preds_collection shape: (Max_Ens, Batch, H, W, 1)
        preds_collection = []
        
        for _ in range(max_ens):
            # 12시간(3 steps) 재귀 예측 수행
            curr = fire_seq[:, 0]
            for i in range(3):
                step = tf.one_hot(tf.fill([batch_size], i), depth=3)
                noise = tf.random.normal((batch_size, LATENT_DIM))
                curr = generator([curr, terrain, tnh_seq[:, i], wind_seq[:, i], step, noise], training=False)
            preds_collection.append(curr) # 최종 12h 결과만 저장
            
        preds_stack = tf.stack(preds_collection, axis=0) # (Max_Ens, B, H, W, 1)
        target_12h = fire_seq[:, 3]
        
        # --- [Step 2] 각 앙상블 스텝별로 성능 계산 ---
        for n in ensemble_steps:
            # 앞에서부터 n개만 잘라서 평균 냄
            subset_preds = preds_stack[:n] # (n, B, H, W, 1)
            mean_pred = tf.reduce_mean(subset_preds, axis=0) # (B, H, W, 1)
            
            # IoU & MSE 계산
            iou = calculate_iou(target_12h, mean_pred).numpy()
            
            mask = get_mask(target_12h, mean_pred)
            mse = masked_mse(target_12h, mean_pred, mask).numpy()
            
            results[n]['iou'].append(iou)
            results[n]['mse'].append(mse)
            
    # 최종 평균 집계
    summary = {n: {'iou': np.mean(results[n]['iou']), 'mse': np.mean(results[n]['mse'])} for n in ensemble_steps}
    return summary

def plot_ensemble_metrics(summary):
    steps = sorted(summary.keys())
    ious = [summary[n]['iou'] for n in steps]
    mses = [summary[n]['mse'] for n in steps]
    
    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    # IoU Plot (Blue)
    color = 'tab:blue'
    ax1.set_xlabel('Number of Ensemble Members (N)')
    ax1.set_ylabel('Mean IoU', color=color, fontweight='bold')
    ax1.plot(steps, ious, marker='o', linestyle='-', color=color, linewidth=2, label='IoU')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.grid(True, alpha=0.3)
    
    # MSE Plot (Red)
    ax2 = ax1.twinx()  
    color = 'tab:red'
    ax2.set_ylabel('Masked MSE', color=color, fontweight='bold')
    ax2.plot(steps, mses, marker='s', linestyle='--', color=color, linewidth=2, label='MSE')
    ax2.tick_params(axis='y', labelcolor=color)
    
    plt.title("Effect of Ensemble Size on Model Performance (12h Prediction)")
    plt.show()
    
    return steps, ious

def visualize_ensemble_steps(generator, dataset, steps=[1, 5, 10], skip=0):
    """
    앙상블 횟수(N)에 따른 예측 결과의 진화를 시각화 (Rebuttal용)
    Terrain | Ground Truth | N=1 | N=5 | N=10 순서로 출력
    """
    # 1. 데이터 가져오기
    for batch in dataset.skip(skip).take(1):
        terrain, fire_seq, tnh_seq, wind_seq = batch
        break
    
    # 배치 내 첫 번째 샘플만 사용
    idx = 0
    target_12h = fire_seq[idx, 3, :, :, 0] # 정답 (12h)
    
    # 2. 최대 횟수만큼 예측 생성 (여기서는 max(steps) = 10)
    max_n = max(steps)
    preds_list = []
    
    print(f"🔄 앙상블 예측 생성 중 (Max N={max_n})...")
    for _ in range(max_n):
        # 12시간(3 steps) 재귀 예측
        curr = fire_seq[idx:idx+1, 0] # 0h 초기화
        for i in range(3):
            step_input = tf.one_hot([i], depth=3)
            noise = tf.random.normal((1, LATENT_DIM))
            # 입력 데이터 슬라이싱
            curr = generator([curr, terrain[idx:idx+1], tnh_seq[idx:idx+1, i], wind_seq[idx:idx+1, i], step_input, noise], training=False)
        preds_list.append(curr[0, :, :, 0]) # (128, 128) 저장
        
    preds_stack = tf.stack(preds_list, axis=0) # (10, 128, 128)
    
    # 3. 시각화 설정 (Terrain + GT + Steps 개수)
    num_cols = 2 + len(steps)
    fig, axes = plt.subplots(1, num_cols, figsize=(5 * num_cols, 5))
    
    # A. Input Terrain
    axes[0].imshow(terrain[idx, :, :, :3])
    axes[0].set_title("Input Terrain", fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    # B. Ground Truth
    axes[1].imshow(target_12h, cmap='inferno')
    axes[1].set_title("Ground Truth (12h)", fontsize=14, fontweight='bold')
    axes[1].axis('off')
    
    # C. Ensemble Steps (N=1, 5, 10)
    for i, n in enumerate(steps):
        ax_idx = i + 2
        
        # 앞쪽 n개만 평균
        subset = preds_stack[:n]
        mean_pred = tf.reduce_mean(subset, axis=0)
        
        # IoU 계산
        iou_val = calculate_iou(target_12h, mean_pred).numpy()
        
        # 시각화
        axes[ax_idx].imshow(mean_pred, cmap='inferno', vmin=0, vmax=1)
        axes[ax_idx].set_title(f"Ensemble N={n}\nIoU: {iou_val:.3f}", fontsize=14, fontweight='bold')
        axes[ax_idx].axis('off')
        
        # 수렴 강조: N=1과 N=10의 차이를 텍스트로 표시하려면 아래 주석 해제
        # if n == 10:
        #     axes[ax_idx].text(5, 120, "Stabilized", color='white', fontweight='bold')

    plt.tight_layout()
    plt.show()


test_tfrecord = "./test_big_128_all_origin.tfrecord"
test_ds_for_ensemble = tf.data.TFRecordDataset([test_tfrecord])
test_ds_for_ensemble = test_ds_for_ensemble.map(_parse_function2, num_parallel_calls=tf.data.AUTOTUNE).batch(1,drop_remainder=True)
test_ds_for_ensemble = test_ds_for_ensemble.prefetch(tf.data.AUTOTUNE)
    # 1. 정량적 분석 (그래프 그리기)
# 시간이 좀 걸릴 수 있으니 test_ds 전체 대신 일부만 하려면 .take(50) 등을 쓰세요.
ensemble_stats = analyze_ensemble_impact(gen_old, test_ds_for_ensemble.take(400), ensemble_steps=[1, 3, 5, 7, 10])
steps, ious = plot_ensemble_metrics(ensemble_stats)

print("\n📈 [Ensemble Analysis Results]")
for n, iou in zip(steps, ious):
    print(f"N={n:<2} | Mean IoU: {iou:.4f}")

# 2. 정성적 분석 (이미지 및 불확실성 맵)
print("\n🎨 Visualizing Uncertainty...")

#test_tfrecord = "./ignition/test_fuck_128_53ea.tfrecord"

print("🎨 Visualizing Ensemble Steps [1, 5, 10]...")
visualize_ensemble_steps(gan.generator, test_ds, steps=[1, 5, 10], skip=0)

In [63]:
def plot_ensemble_metrics(summary):
    steps = sorted(summary.keys())
    ious = [summary[n]['iou'] for n in steps]
    mses = [summary[n]['mse'] for n in steps]
    
    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    # IoU Plot (Blue)
    color = 'tab:blue'
    ax1.set_xlabel('Ensemble size')
    ax1.set_ylabel('Mean IoU', color=color)
    ax1.plot(steps, ious, marker='o', linestyle='-', color=color, linewidth=2, label='IoU')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.grid(True, alpha=0.3)
    
    # MSE Plot (Red)
    ax2 = ax1.twinx()  
    color = 'tab:red'
    ax2.set_ylabel('MSE', color=color)
    ax2.plot(steps, mses, marker='s', linestyle='--', color=color, linewidth=2, label='MSE')
    ax2.tick_params(axis='y', labelcolor=color)
    
    plt.show()
    
    return steps, ious

steps, ious = plot_ensemble_metrics(ensemble_stats)

In [26]:
import time

def measure_inference_time(generator, dataset, ensemble_steps=[1, 5, 10, 20, 50]):
    """
    앙상블 횟수(N)에 따른 추론 시간 측정
    """
    # 1. 데이터 샘플 준비 (배치 1개)
    for batch in dataset.take(1):
        terrain, fire_seq, tnh_seq, wind_seq = batch
        break
    
    # 배치 사이즈 확인
    B = tf.shape(terrain)[0]
    print(f"⏱️ 시간 측정 시작 (Batch Size: {B})...")
    
    # 2. GPU Warm-up (초기 실행 오버헤드 제거용)
    print("🔥 GPU 예열 중...")
    curr = fire_seq[:, 0]
    step = tf.one_hot(tf.fill([B], 0), depth=3)
    noise = tf.random.normal((B, LATENT_DIM))
    _ = generator([curr, terrain, tnh_seq[:, 0], wind_seq[:, 0], step, noise], training=False)
    
    # 3. 측정 루프
    times = []
    
    for n in ensemble_steps:
        start_time = time.time()
        
        # N번 앙상블 수행 (실제 예측 로직과 동일하게 구성)
        preds = []
        for _ in range(n):
            curr_pred = fire_seq[:, 0]
            # 12시간(3 steps) 재귀 예측
            for i in range(3):
                step_idx = tf.one_hot(tf.fill([B], i), depth=3)
                noise = tf.random.normal((B, LATENT_DIM))
                curr_pred = generator([curr_pred, terrain, tnh_seq[:, i], wind_seq[:, i], step_idx, noise], training=False)
            
            # 동기화: GPU 연산이 끝날 때까지 대기 (결과를 리스트에 넣음)
            preds.append(curr_pred)
            
        # 마지막 연산 완료 시점 확실히 하기 위해 값을 하나 꺼내봄
        _ = preds[-1].numpy()
        
        end_time = time.time()
        elapsed = end_time - start_time
        times.append(elapsed)
        print(f"  - N={n:<2}: {elapsed:.4f} sec")
        
    return ensemble_steps, times

def plot_time_vs_performance(n_steps, times, iou_summary=None):
    """
    시간(Time)과 성능(IoU)을 하나의 그래프에 그려 Trade-off를 시각화
    iou_summary: 앞선 analyze_ensemble_impact 함수의 결과 딕셔너리 (옵션)
    """
    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    # 1. 시간 그래프 (막대 or 선)
    color = 'tab:red'
    ax1.set_xlabel('Ensemble Size (N)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Inference Time (sec)', color=color, fontsize=12, fontweight='bold')
    ax1.plot(n_steps, times, marker='s', linestyle='--', color=color, label='Time Cost')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.grid(True, alpha=0.3)
    
    # 2. 성능 그래프 (있을 경우 Twin axis)
    if iou_summary is not None:
        # iou_summary 키가 n_steps에 있는 것만 추출
        ious = [iou_summary[n]['iou'] for n in n_steps if n in iou_summary]
        # n_steps와 iou 개수가 맞는지 확인
        valid_steps = [n for n in n_steps if n in iou_summary]
        
        if len(ious) > 0:
            ax2 = ax1.twinx()
            color = 'tab:blue'
            ax2.set_ylabel('Mean IoU', color=color, fontsize=12, fontweight='bold')
            ax2.plot(valid_steps, ious, marker='o', linestyle='-', color=color, linewidth=2, label='Performance (IoU)')
            ax2.tick_params(axis='y', labelcolor=color)
            
            # 교차점(Trade-off Point) 강조 텍스트
            plt.title("Trade-off: Inference Time vs Performance", fontsize=14)
    else:
        plt.title("Inference Time by Ensemble Size", fontsize=14)
        
    plt.tight_layout()
    plt.show()

# === 실행 ===
# 1. 시간 측정 (N을 다양하게 설정)
n_steps_list = [1, 5, 10, 20, 30]
steps, times = measure_inference_time(gen_old, test_ds_for_ensemble.take(100), ensemble_steps=n_steps_list)

# 2. 그래프 그리기 (앞서 계산한 ensemble_stats가 있다면 같이 넣어주세요)
# ensemble_stats가 없으면 None으로 넣으시면 시간만 나옵니다.
# 앞선 코드의 ensemble_stats 변수가 메모리에 있다면 아래 주석 해제:
plot_time_vs_performance(steps, times, iou_summary=ensemble_stats)

In [27]:
import os
import time
import tracemalloc  # Python 내장 메모리 추적 라이브러리

def measure_deployment_full_specs(model, dataset, ensemble_size=5):
    """
    [Rebuttal용] Edge Device 배포 가능성 검증 (시간, 파라미터, 저장공간, 시스템 메모리)
    """
    print(f"📊 [Edge Deployment Feasibility Check] (Ensemble N={ensemble_size})")
    
    # 1. 모델 파라미터 및 용량 (Static Specs)
    total_params = model.count_params()
    # 임시 저장하여 실제 파일 크기 확인
    temp_path = 'temp_model_deploy.keras'
    model.save(temp_path)
    file_size_mb = os.path.getsize(temp_path) / (1024 * 1024)
    if os.path.exists(temp_path): os.remove(temp_path)
    
    print(f"  [1] Model Scale:")
    print(f"    - Parameters: {total_params:,}")
    print(f"    - Storage Size: {file_size_mb:.2f} MB")

    # 2. 데이터 준비
    for batch in dataset.take(1):
        terrain, fire_seq, tnh_seq, wind_seq = batch
        break
    B = tf.shape(terrain)[0]
    curr = fire_seq[:, 0]

    # --- 추론 함수 ---
    def run_inference_loop():
        # 실제 엣지 환경처럼 N번 반복 추론
        preds = []
        for _ in range(ensemble_size):
            c = curr
            for i in range(3): # 12h
                step = tf.one_hot(tf.fill([B], i), depth=3)
                noise = tf.random.normal((B, LATENT_DIM))
                c = model([c, terrain, tnh_seq[:, i], wind_seq[:, i], step, noise], training=False)
            preds.append(c)
        return preds[-1]

    # 3. GPU 성능 측정 (High-End)
    print(f"  [2] GPU Performance (Server-grade):", end="")
    _ = run_inference_loop() # Warm-up
    start = time.time()
    _ = run_inference_loop()
    gpu_time = time.time() - start
    print(f" {gpu_time:.4f} sec")

    # 4. CPU 성능 및 메모리 측정 (Edge Device Simulation)
    print(f"  [3] CPU Performance (Edge Device Simulation):")
    cpu_time = None
    peak_ram_mb = None
    
    try:
        with tf.device('/CPU:0'):
            # Warm-up (메모리 할당 안정화)
            _ = run_inference_loop()
            
            # --- 측정 시작 ---
            tracemalloc.start() # 메모리 추적 시작
            start = time.time()
            
            _ = run_inference_loop() # 추론 실행
            
            cpu_time = time.time() - start
            current, peak = tracemalloc.get_traced_memory() # 메모리 사용량 확인
            tracemalloc.stop() # 추적 종료
            
            peak_ram_mb = peak / (1024 * 1024) # Byte -> MB
            
        print(f"    - Inference Time: {cpu_time:.4f} sec")
        print(f"    - Peak System RAM: {peak_ram_mb:.2f} MB (Additional)")
        
    except Exception as e:
        print(f"    (CPU 측정 실패: {e})")

    return {
        "params": total_params,
        "size_mb": file_size_mb,
        "gpu_time": gpu_time,
        "cpu_time": cpu_time,
        "ram_mb": peak_ram_mb
    }

# === 측정 실행 (N=5 기준) ===
specs = measure_deployment_full_specs(gen_old, test_ds_for_ensemble, ensemble_size=5)

In [89]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
# Time labels
time_labels = ['4h', '8h', '12h']
try:
    font_path = '/Library/Fonts'
    fm.fontManager.addfont(font_path)
    plt.rcParams['font.family'] = 'Times New Roman'
except FileNotFoundError:
    print("Warning: Times New Roman font not found. Using default serif font.")
    plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 14
plt.rcParams['axes.titlesize'] = 26
plt.rcParams['axes.labelsize'] = 20
plt.rcParams['legend.fontsize'] = 18
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16

# === CGAN-based Data (from Screenshot 1) ===
# === CGAN-based Data (from image_4eeb5e.png) ===
pre_train_train_mse = [0.27710, 0.19814, 0.07653]
pre_train_train_ssim = [0.98472, 0.97570, 0.95224]
pre_train_train_perim = [0.27777, 0.22845, 0.19431]

pre_train_test_mse = [0.27246, 0.18967, 0.07784]
pre_train_test_ssim = [0.98108, 0.96900, 0.94629]
pre_train_test_perim = [0.22332, 0.20938, 0.16495]

# === TL-CGAN-based model Data (from image_4eeb5e.png) ===
post_train_train_mse = [0.08769, 0.06890, 0.01756]
post_train_train_ssim = [0.99335, 0.98667, 0.97486]
post_train_train_perim = [0.06541, 0.07426, 0.05947]

post_train_test_mse = [0.17180, 0.11060, 0.03315]
post_train_test_ssim = [0.98564, 0.97691, 0.96141]
post_train_test_perim = [0.11590, 0.10434, 0.06593]


# Marker and Style configurations
# Solid: Train, Dashed: Test
# Blue shades: CGAN-based, Red shades: TL-CGAN-based model

PRE_TRAIN_COLOR   = '#0d47a1'  # 진한 남색
PRE_TEST_COLOR    = '#1976d2'  # 선명한 파랑
POST_TRAIN_COLOR = '#b71c1c'  # 진한 붉은색
POST_TEST_COLOR  = '#e53935'  # 선명한 붉은색

# 1. Mean Squared Error (MSE)
plt.figure(figsize=(8, 10))
plt.plot(time_labels, pre_train_train_mse, marker='o', linestyle='-', color=PRE_TRAIN_COLOR, label='CGAN Train')
plt.plot(time_labels, pre_train_test_mse, marker='d', linestyle='--', color=PRE_TEST_COLOR, label='CGAN Test')
plt.plot(time_labels, post_train_train_mse, marker='o', linestyle='-', color=POST_TRAIN_COLOR, label='TL-CGAN Train')
plt.plot(time_labels, post_train_test_mse, marker='d', linestyle='--', color=POST_TEST_COLOR, label='TL-CGAN Test')
plt.ylabel('Mean Squared Error (MSE)')
plt.xlabel('Target Prediction Time')
plt.ylim(0,1)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(loc='upper right')

# 2. Structural Similarity Index (SSIM)
plt.figure(figsize=(8, 10))
plt.plot(time_labels, pre_train_train_ssim, marker='o', linestyle='-', color=PRE_TRAIN_COLOR, label='CGAN Train')
plt.plot(time_labels, pre_train_test_ssim, marker='d', linestyle='--', color=PRE_TEST_COLOR, label='CGAN Test')
plt.plot(time_labels, post_train_train_ssim, marker='o', linestyle='-', color=POST_TRAIN_COLOR, label='TL-CGAN Train')
plt.plot(time_labels, post_train_test_ssim, marker='d', linestyle='--', color=POST_TEST_COLOR, label='TL-CGAN Test')
plt.ylabel('Structural Similarity Index (SSIM)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlabel('Target Prediction Time')
plt.ylim(0,1.0)
plt.legend(loc='lower right')

plt.figure(figsize=(8, 10))

plt.plot(time_labels, pre_train_train_perim,
         marker='o', linestyle='--', label='CGAN Train',
         color=PRE_TRAIN_COLOR, markersize=8)
plt.plot(time_labels, post_train_train_perim,
         marker='*', linestyle='-', label='TL-CGAN Train',
         color=POST_TRAIN_COLOR, linewidth=2, markersize=10)

plt.ylabel('Boundary MAE (B-MAE)')
plt.xlabel('Target Prediction Time')
plt.legend()
plt.ylim(0,0.3)
plt.grid(True, linestyle='--', alpha=0.6)

plt.savefig("B_MAE_train_comparison_final.png", dpi=600, bbox_inches='tight')
plt.show()

plt.figure(figsize=(8, 10))

plt.plot(time_labels, pre_train_test_perim,
         marker='o', linestyle='--', label='CGAN Test',
         color=PRE_TRAIN_COLOR, markersize=8)
plt.plot(time_labels, post_train_test_perim,
         marker='*', linestyle='-', label='TL-CGAN Test',
         color=POST_TRAIN_COLOR, linewidth=2, markersize=10)

plt.ylabel('Boundary MAE (B-MAE)')
plt.xlabel('Target Prediction Time')
plt.ylim(0,0.3)
plt.legend(loc="upper right")
plt.grid(True, linestyle='--', alpha=0.6)

plt.savefig("B_MAE_train_comparison_final.png", dpi=600, bbox_inches='tight')
plt.show()


# 3. Boundary MAE (B-MAE)
plt.figure(figsize=(8, 10))
plt.plot(time_labels, pre_train_train_perim, marker='o', linestyle='-', color=PRE_TRAIN_COLOR, label='CGAN Train')
plt.plot(time_labels, pre_train_test_perim, marker='d', linestyle='--', color=PRE_TEST_COLOR, label='CGAN-based model Test')
plt.plot(time_labels, post_train_train_perim, marker='o', linestyle='-', color=POST_TRAIN_COLOR, label='TL-CGAN-based model Train')
plt.plot(time_labels, post_train_test_perim, marker='d', linestyle='--', color=POST_TEST_COLOR, label='TL-CGAN-based model Test')
plt.ylabel('Boundary MAE (B-MAE)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlabel('Target Prediction Time')
plt.legend()

plt.tight_layout()
plt.savefig('performance_comparison_all.png')